# 用 Unsloth 在 AMD Radeon GPU 上微调大语言模型（LoRA SFT）

随着开源大语言模型（LLM）的普及，把一个通用模型「微调」成更贴合自己业务/数据的专属模型，已经成为很多团队的刚需。但全参数微调显存开销大、迭代慢，对本地硬件不太友好。

[Unsloth](https://github.com/unslothai/unsloth) 通过优化显存占用和训练速度，让 LLM 微调更容易在本地硬件上跑起来。本教程带你在 **AMD Radeon™ GPU（ROCm™ 软件栈）** 上，用 **LoRA + 监督微调（SFT）** 完整跑通一次「环境配置 → 加载模型 → 准备数据 → 训练 → 推理 → 保存」的端到端流程。

示例使用 `unsloth/gemma-4-E4B-it` 作为基座模型，数据集取 `mlabonne/FineTome-100k` 的一个子集，做一个简短的 LoRA SFT。整个流程很轻量，方便你照着改成自己的数据和模型。


## 你会学到

学完本教程，你将能够：

1. 在 AMD Radeon GPU（ROCm 软件栈）上确认硬件、并配置好 Unsloth 微调环境。
2. 用 Unsloth + LoRA 对一个大语言模型做监督微调（SFT），理解加载模型、准备数据、训练、推理的完整链路。
3. 把微调结果保存下来（LoRA 适配器 / 合并后的完整模型 / GGUF），并了解后续部署到 vLLM、llama.cpp 的方式。


## 为什么用 Unsloth？

Unsloth 通过重写计算内核，相比标准方案**显著降低显存占用、加快训练速度**，因此更适合在本地/单卡硬件上做 LLM 微调。

本教程采用 **基于 LoRA 的 SFT**：基座模型的大部分权重保持冻结，只训练一小撮额外的「适配器（adapter）」权重。相比全参数微调，它更省显存、迭代更快，非常适合本地开发与快速试验。

> Unsloth 也支持 QLoRA、强化学习等更多训练方式。本教程从最简单的一条路径切入：一个小规模、可运行、易理解、好扩展的 LoRA 微调示例。


## 前置条件

本教程在以下环境开发并验证。

### 操作系统

- **Ubuntu 24.04** 或 **Ubuntu 22.04**（Radeon 上的 ROCm 对发行版有版本要求，请以 [ROCm Radeon 兼容性矩阵](https://rocm.docs.amd.com/projects/radeon/en/latest/docs/compatibility/native_linux/native_linux_compatibility.html) 为准）。

### 硬件

- **AMD Radeon GPU**：受支持型号包括 RX 7900 XTX / XT / GRE、RX 9070 XT / GRE / 9070、RX 9060 XT、Radeon AI PRO R9700、PRO W7900 / W7800 等。请在 [ROCm Radeon 兼容性矩阵](https://rocm.docs.amd.com/projects/radeon/en/latest/docs/compatibility/native_linux/native_linux_compatibility.html) 中确认你的卡。
- **显存**：本示例为 LoRA 微调，显存需求远低于全参数微调，建议 **16GB 显存及以上**。换更大的模型或更长的序列长度时，请相应预留更多显存。

> **说明**：如果你在 Radeon Cloud 的 Jupyter 实例里运行，环境通常已预装好 ROCm 与 PyTorch，「一、准备 Python 环境」以及「二」中安装 PyTorch 的步骤可跳过；但「二、安装依赖」里的**安装 Unsloth**仍需执行，随后再进行「三、验证环境」。


## 检查 GPU 与 ROCm

先用 `amd-smi` 确认 GPU 能被识别、ROCm 已就绪。

> **注意**：ROCm 7.x 推荐使用 `amd-smi`（旧版命令 `rocm-smi` 也仍可用）。


In [ ]:
!amd-smi

![amd-smi.png](../assets/amd-smi.png)

## 一、准备 Python 环境（手动安装时才需要）

下面用 `venv` 创建并激活一个 Python 虚拟环境。这些命令需要在**终端**里执行（而不是 notebook 单元里）。

先授予当前用户访问 GPU 设备的权限（执行后需重新登录生效）：

```bash
sudo usermod -aG render,video $LOGNAME
```

创建并激活虚拟环境：

```bash
sudo apt update
sudo apt install -y python3-venv
python3 -m venv unsloth-env
source unsloth-env/bin/activate
```

> 如果你的系统已预装 ROCm 版 PyTorch，创建 venv 时可加 `--system-site-packages` 复用系统已装好的 PyTorch：
> `python3 -m venv unsloth-env --system-site-packages`


## 二、安装依赖

### PyTorch（ROCm 版，Radeon Cloud 上已预装，可跳过）

如果环境里还没有 ROCm 版 PyTorch，请按 [ROCm Radeon PyTorch 安装文档](https://rocm.docs.amd.com/projects/radeon/en/latest/docs/install/native_linux/install-pytorch.html) 安装官方 wheel（**建议用 [repo.radeon.com](https://repo.radeon.com) 上的 wheel，而不是 pytorch.org 的版本**）。

### 安装 Unsloth（Radeon Cloud 上也需执行）

在虚拟环境里安装 Unsloth（AMD 版）。

> **transformers 版本**：Gemma 4 需要 `transformers>=5.2.0,<=5.5.0`；安装时若出现 `vllm` 版本冲突提示，可忽略。


In [ ]:
!pip install "unsloth[amd] @ git+https://github.com/unslothai/unsloth.git"
%pip install "transformers>=5.2.0,<=5.5.0"

> **关于 `bitsandbytes` 警告**：导入 Unsloth 时，它可能探测 `bitsandbytes` 加速路径，在某些 ROCm 版本上出现 `bitsandbytes library load error: Configured ROCm binary not found` 之类的提示。本教程用的是标准 LoRA 微调（优化器为 `adamw_torch`），**不依赖 `bitsandbytes` 优化器，也不用 4-bit QLoRA**，因此该提示可以安全忽略。


## 三、验证环境

依次运行下面的单元，确认 PyTorch 能识别 GPU、且 Unsloth 相关依赖可正常导入。

> **注意**：ROCm 上的 PyTorch 复用了 CUDA 的 API 命名，因此 `torch.cuda.is_available()`、`torch.cuda.get_device_name()` 在 AMD GPU 上同样适用（底层走 HIP）。


In [ ]:
!python3 -c 'import torch' 2> /dev/null && echo 'Success' || echo 'Failure'

In [ ]:
!python3 -c 'import torch; print(torch.cuda.is_available())'

In [ ]:
!python3 -c "import torch; print('device name [0]:', torch.cuda.get_device_name(0))"

验证 Unsloth 及训练相关库能正常导入：


In [ ]:
import torch
from datasets import load_dataset
from transformers import TextStreamer
from unsloth import FastModel
from unsloth.chat_templates import (
    get_chat_template,
    standardize_data_formats,
    train_on_responses_only,
)
from trl import SFTTrainer, SFTConfig

print(f"PyTorch version: {torch.__version__}")
print(f"ROCm available: {torch.cuda.is_available()}")
print("PASS: 所有依赖导入成功")

![import-success.png](../assets/import-success.png)

## 下载模型（魔搭 ModelScope）

本教程的基座模型通过 [魔搭社区 ModelScope](https://modelscope.cn) 下载到本地，后续直接从本地目录加载。先安装 ModelScope，再下载模型：

In [ ]:
%pip install -U modelscope addict simplejson sortedcontainers
!modelscope download --model unsloth/gemma-4-E4B-it --local_dir ./models/gemma-4-E4B-it

> 下载完成后，下面「关键配置参数」里的 `MODEL_NAME` 已指向该本地目录 `./models/gemma-4-E4B-it`。

## 关键配置参数

下面集中定义本次微调用到的关键参数，后续单元会直接引用。你可以按需修改：


In [ ]:
MODEL_NAME   = "./models/gemma-4-E4B-it"   # 基座模型（上一步从魔搭下载的本地目录）
MAX_SEQ_LEN  = 1024                        # 最大序列长度
DATASET_NAME = "mlabonne/FineTome-100k"    # 数据集（魔搭 ModelScope）
OUTPUT_DIR   = "gemma_4_lora"              # LoRA 适配器保存目录

## 四、加载基座模型

用 Unsloth 的 `FastModel` 加载基座模型。这里**不开启 4-bit 量化**（`load_in_4bit=False`），因为我们在 AMD ROCm 上走标准 LoRA，不依赖 `bitsandbytes`。


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = False,   # AMD ROCm 上用标准 LoRA，关闭 4-bit
    full_finetuning = False,   # 用 LoRA，而非全参数微调
)

![model-load.png](../assets/model-load.png)

## 五、添加 LoRA 适配器

给模型挂上 LoRA 适配器：基座权重冻结，只训练这些小规模的适配器。我们同时为**语言层、注意力模块、MLP 模块**添加适配器。


In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_language_layers   = True,   # 语言层
    finetune_attention_modules = True,   # 注意力模块
    finetune_mlp_modules       = True,   # MLP 模块
    r            = 8,        # LoRA 秩，越大容量越高、显存占用越多
    lora_alpha   = 8,
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)

## 六、准备数据集

我们从 [魔搭社区 ModelScope](https://modelscope.cn) 加载 `mlabonne/FineTome-100k` 的一个子集（取前 2000 条作演示，跑得更快），并：

- 转成对话（chat）格式；
- 套用 **Gemma-4 chat 模板**；
- 清理重复的 BOS（句首）标记。


In [ ]:
from modelscope.msdatasets import MsDataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats

# 套用 Gemma-4 chat 模板
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")

# 从魔搭(ModelScope)加载数据集
dataset = MsDataset.load(DATASET_NAME, split = "train")
dataset = dataset.select(range(2000))   # 取子集做演示（删掉这行即用全量）

# 统一成标准对话字段格式
dataset = standardize_data_formats(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")   # 去掉重复的 BOS，避免句首出现两个
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset)
print("--- 一条样本预览 ---")
print(dataset[0]["text"][:600])

![dataset-prep.png](../assets/dataset-prep.png)

## 七、训练模型

用 TRL 的 `SFTTrainer` 做监督微调。为了演示，这里只跑 **50 步（`max_steps=50`）**、小批量 + 梯度累积。真正训练时可以改用 `num_train_epochs` 跑完整数据集。

> **AMD 关键点**：优化器用 `optim="adamw_torch"`（PyTorch 自带），**不依赖 `bitsandbytes`**——这正是它能在 AMD ROCm 上稳定跑起来的原因。


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 50,      # 演示用；正式训练可改 num_train_epochs
        learning_rate               = 2e-4,
        logging_steps               = 1,
        optim                       = "adamw_torch",  # AMD ROCm：用 torch 自带优化器
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)

![dataset-tokenize.png](../assets/dataset-tokenize.png)

只对「模型回复」部分计算损失（response-only loss masking），让模型专注学习如何回答，而不是去拟合用户的提问：


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)

开始训练。训练过程中会逐步打印 loss 日志：


In [ ]:
trainer_stats = trainer.train()

![train-loss.png](../assets/train-loss.png)

## 八、推理测试

训练完成后，用微调过的模型做一次快速生成，直观感受效果：


In [ ]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": [
        {"type": "text", "text": "用一句话解释什么是 LoRA 微调。"},
    ]},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    tokenize       = True,
    return_dict    = True,
    return_tensors = "pt",
).to("cuda")

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    max_new_tokens = 128,
    streamer       = streamer,
    temperature    = 0.7,
    top_p          = 0.95,
)

![infer-output.png](../assets/infer-output.png)

## 九、保存与部署

### 1）保存 LoRA 适配器（本地）

只保存训练出来的 LoRA 适配器（体积很小，加载时需配合原基座模型一起使用）：


In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA 适配器已保存到: {OUTPUT_DIR}")

![save-lora.png](../assets/save-lora.png)

### 2）保存合并后的完整模型（用于 vLLM）

把 LoRA 适配器合并进基座模型，得到一个完整模型，便于用 vLLM 部署：


In [ ]:
model.save_pretrained_merged("gemma-4-finetune", tokenizer)

![save-merged.png](../assets/save-merged.png)

### 3）导出 GGUF（用于 llama.cpp / Ollama）

把模型导出成 GGUF 格式，便于在 llama.cpp、Ollama 等本地推理引擎上运行。

> **⚠️ 关于网络（重要）**：Unsloth 首次导出 GGUF 时，会尝试从 GitHub **下载预编译包或克隆并编译 `llama.cpp`**。在带有 **TLS 拦截代理（自签名证书）** 的环境（如本 Radeon Cloud）里，Python 下载会因证书校验失败（`SSL: CERTIFICATE_VERIFY_FAILED: self-signed certificate`），并可能误报 `You do not have internet connection!`。
>
> **解决办法**：先**手动把 `llama.cpp` 克隆并编译到 `~/.unsloth/llama.cpp`**。这样 Unsloth 会检测到本地已存在的 llama.cpp，直接跳过下载与联网检测。下面几个单元依次完成：**克隆 → 装编译依赖 → 编译 → 导出**。

In [ ]:
import os

# 记住当前工作目录，导出时再切回来
_WORKDIR = os.getcwd()

# 克隆 llama.cpp 到 Unsloth 期望的目录 ~/.unsloth/llama.cpp
os.makedirs(os.path.expanduser("~/.unsloth"), exist_ok=True)
os.chdir(os.path.expanduser("~/.unsloth"))
!git clone --recursive https://github.com/ggerganov/llama.cpp

In [ ]:
# 安装编译依赖（走内网 apt 源；如提示权限不足，前面加 sudo）
!apt-get update && apt-get install -y build-essential cmake libcurl4-openssl-dev

In [ ]:
# 编译 llama.cpp，并把生成的 llama-* 二进制拷到仓库根目录（Unsloth 会在这里查找）
!cmake llama.cpp -B llama.cpp/build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=OFF -DLLAMA_CURL=OFF
!cmake --build llama.cpp/build --config Release -j --clean-first --target llama-quantize llama-cli llama-gguf-split
!cp llama.cpp/build/bin/llama-* llama.cpp/

In [ ]:
# 切回原工作目录再导出。llama.cpp 已就绪，Unsloth 会打印
# “llama.cpp found in the system. Skipping installation.”，不再联网。
os.chdir(_WORKDIR)
model.save_pretrained_gguf("gemma_4_finetune", tokenizer, quantization_method="Q8_0")

![gguf-success.png](../assets/gguf-success.png)

> 导出完成后会在 `gemma_4_finetune_gguf/` 下得到量化模型 `*.Q8_0.gguf`。若基座是**多模态模型**（如 Gemma 4），还会额外生成 `*-mmproj.gguf`（视觉投影器，供带图输入时使用）。

## 进阶与下一步

- 把演示用的 `train[:2000]` / `max_steps=50` 换成全量数据、`num_train_epochs`，做一次完整训练。
- 换成你自己的数据集（整理成对话格式即可），微调出贴合业务的专属模型。
- 调整超参数（LoRA 秩 `r`、学习率、序列长度等）观察效果变化。
- 用 vLLM 或 llama.cpp 部署微调后的模型。
- 尝试 QLoRA 以进一步降低显存占用。

更多资料：

- [Unsloth 文档](https://docs.unsloth.ai)
- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [Unsloth 微调指南](https://docs.unsloth.ai/get-started/fine-tuning-llms-guide)
- [AMD Playbook：Fine-tuning LLMs with Unsloth](https://github.com/amd/playbooks/blob/main/playbooks/supplemental/unsloth-llms-finetuning/README.md)


## 课后测验

**第 1 题：** 在 **ROCm 7.x** 的 AMD Radeon GPU 上，查看 GPU 状态与 ROCm 版本，推荐使用下面哪个命令？

- A. `nvidia-smi`
- B. `amd-smi`
- C. `dxdiag`
- D. `top`

**第 2 题：** 本教程在 **AMD Radeon（ROCm）** 上做 LoRA 微调时，`SFTConfig` 里把优化器设为 `optim="adamw_torch"`，主要原因是？

- A. 它比所有其他优化器收敛都快
- B. 它是 Unsloth 唯一支持的优化器
- C. 标准 LoRA 微调不依赖 `bitsandbytes`，用 PyTorch 自带优化器可在 AMD ROCm 上稳定运行
- D. 它能自动把模型量化到 4-bit

**第 3 题：** 关于在 **AMD Radeon GPU** 上用 Unsloth 做 **LoRA 微调** 的显存特点，下列说法正确的是？

- A. LoRA 微调会训练模型的全部参数，因此比全参数微调更吃显存
- B. LoRA 冻结基座权重、只训练少量适配器权重，相比全参数微调更省显存，更适合单卡/本地运行
- C. LoRA 只能在 NVIDIA GPU 上运行，AMD Radeon 不支持
- D. LoRA 微调必须开启 4-bit 量化（QLoRA）才能运行

---

> **参考答案：** 1-B，2-C，3-B
